In [ ]:
# ─────────────────────────────────────────────
# [C1] ⚙️ 데이터 준비
# 최초 1회 다운로드 → data/ 폴더에 저장 (이후 오프라인)
# ─────────────────────────────────────────────
import urllib.request, zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

def fetch_uci(url, zip_name, member):
    """UCI 정적 저장소의 zip을 내려받아 data/에 풀고, CSV 경로를 돌려줍니다."""
    csv_path = DATA_DIR / member
    if not csv_path.exists():
        zip_path = DATA_DIR / zip_name
        if not zip_path.exists():
            print(f"내려받는 중… {zip_name}")
            urllib.request.urlretrieve(url, zip_path)
        with zipfile.ZipFile(zip_path) as z:
            z.extract(member, DATA_DIR)
    return csv_path

shoppers = pd.read_csv(fetch_uci(
    "https://archive.ics.uci.edu/static/public/468/online+shoppers+purchasing+intention+dataset.zip",
    "online_shoppers.zip", "online_shoppers_intention.csv"))

NUM_COLS = ["Administrative", "Administrative_Duration", "Informational",
            "Informational_Duration", "ProductRelated", "ProductRelated_Duration",
            "BounceRates", "ExitRates", "PageValues", "SpecialDay"]

X = shoppers[NUM_COLS]
y = shoppers["Revenue"].astype(int)

print(f"쇼핑 세션 데이터: {shoppers.shape[0]:,}행 × {shoppers.shape[1]}열")
print(f"구매 전환율(양성 비율): {y.mean():.4f}")
print("\n→ 준비 완료. 이제 여러분 차례입니다.")

내려받는 중… online_shoppers.zip
쇼핑 세션 데이터: 12,330행 × 18열
구매 전환율(양성 비율): 0.1547

→ 준비 완료. 이제 여러분 차례입니다.


In [ ]:
# [C2] 문제 1. 혼동행렬로 0.89를 쪼갠다
# ⌨️ 문제 1 — 정확도 하나를 네 칸으로 쪼개기
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (confusion_matrix, accuracy_score, precision_score,
                             recall_score, f1_score)

# 1) 분리 — 지난 순서와 동일한 분할
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 2) 학습
model = RandomForestClassifier(n_estimators=300, min_samples_leaf=20, random_state=42)
model.fit(X_tr, y_tr)

# 3) 예측 및 혼동행렬
pred = model.predict(X_te)
tn, fp, fn, tp = confusion_matrix(y_te, pred).ravel()

print("혼동행렬 (Confusion Matrix)")
print(f"  TN (실제 안 삼 / 예측 안 삼) = {tn}")
print(f"  FP (실제 안 삼 / 예측 삼)   = {fp}")
print(f"  FN (실제 삼   / 예측 안 삼) = {fn}")
print(f"  TP (실제 삼   / 예측 삼)   = {tp}")

# 4) 지표 4종
acc = accuracy_score(y_te, pred)
prec = precision_score(y_te, pred)
rec = recall_score(y_te, pred)
f1 = f1_score(y_te, pred)

print("\n지표 4종")
print(f"  정확도(Accuracy)  = {acc:.4f}")
print(f"  정밀도(Precision) = {prec:.4f}")
print(f"  재현율(Recall)    = {rec:.4f}")
print(f"  F1 Score          = {f1:.4f}")

# 5) 놓친 구매 세션
actual_pos = y_te.sum()
print(f"\n실제 구매 세션 수 = {actual_pos}건")
print(f"놓친 구매 세션(FN) = {fn}건")
print(f"놓친 비율 = {fn/actual_pos:.4f} ({fn/actual_pos*100:.1f}%)")

혼동행렬 (Confusion Matrix)
  TN (실제 안 삼 / 예측 안 삼) = 2002
  FP (실제 안 삼 / 예측 삼)   = 82
  FN (실제 삼   / 예측 안 삼) = 186
  TP (실제 삼   / 예측 삼)   = 196

지표 4종
  정확도(Accuracy)  = 0.8913
  정밀도(Precision) = 0.7050
  재현율(Recall)    = 0.5131
  F1 Score          = 0.5939

실제 구매 세션 수 = 382건
놓친 구매 세션(FN) = 186건
놓친 비율 = 0.4869 (48.7%)


**정확도 0.89는 착시였습니다**

- 전체 2,466건 중 2,213건(TN+TP)을 맞혔다는 게 정확도인데, 이 중 대부분(2,002건)이 그냥 "안 사는 사람을 안 산다"고 맞춘 것입니다.
- 실제로 어려운 일 — "살 사람을 산다고 맞추는 것" — 은 382건 중 196건밖에 못 잡았습니다.

**재현율 0.5131이 핵심 문제입니다**

- 실제 구매 세션 382건 중 **186건(48.7%)을 놓쳤습니다.** 거의 절반이죠.
- 마케팅팀이 "구매 가능성 있는 세션에 리타겟팅 광고를 쏘겠다"는 목적이었다면, 이 모델은 잠재 구매자의 절반을 그냥 놔주는 셈입니다. 정확도 89%라는 숫자만 보고 "잘 작동한다"고 보고했다면 완전히 다른 그림을 놓치는 겁니다.

**정밀도 0.7050은 반대로 괜찮은 편입니다**

- 모델이 "이 사람 산다"고 예측한 278건(FP 82 + TP 196) 중 196건이 실제로 맞았습니다. 즉 "구매할 것"이라고 찍은 사람들은 70%가 실제로 삽니다.
- 이건 이 모델이 **보수적으로 예측**한다는 뜻입니다 — 확신이 강한 케이스만 "구매"로 찍고, 애매한 케이스는 다 "안 삼"으로 분류해버리는 성향이죠.

**왜 이런 불균형이 생겼나**

- 원본 데이터의 구매 전환율이 15% 안팎(불균형 클래스)이라, 모델이 "안 삼"으로만 찍어도 기본 정확도가 높게 나오는 구조입니다.
- `RandomForestClassifier`의 기본 판단 기준(threshold 0.5)이 다수 클래스(안 삼) 쪽으로 편향되기 쉽습니다.

**실무적 시사점**

마케팅팀 목적이 "구매 세션을 최대한 많이 포착"이라면 —
- 재현율을 올려야 하는데, 보통 **분류 임계값(threshold)을 낮추거나**, `class_weight='balanced'`를 주거나, 정밀도-재현율 트레이드오프를 조정해서 재현율을 끌어올리는 방향으로 갑니다. 대신 정밀도(0.705)는 떨어지고 FP가 늘어나겠죠 — "안 살 사람에게도 광고비를 쓰는" 비용이 생깁니다.

In [ ]:
# [C3] 문제 2. 지표 5종을 교차 검증으로 한 번에 잰다
# ⌨️ 문제 2 — cross_validate로 다섯 지표를 동시에
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.metrics import roc_auc_score

# 1) 5-Fold 교차검증으로 지표 5종
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = ["accuracy", "precision", "recall", "f1", "average_precision"]
results = cross_validate(model, X, y, cv=cv, scoring=scoring)

# 2) 평균 ± 표준편차 출력
print("=== 5-Fold 교차검증 결과 (평균 ± 표준편차) ===")
summary = {}
for metric in scoring:
    scores = results[f"test_{metric}"]
    mean, std = scores.mean(), scores.std()
    summary[metric] = (mean, std)
    print(f"  {metric:20s} = {mean:.4f} ± {std:.4f}")

# 3) AP vs 기준선(양성 비율), ROC-AUC
baseline = y.mean()
ap_mean = summary["average_precision"][0]
print(f"\n기준선(무작위 정밀도) = 양성 비율 = {baseline:.4f}")
print(f"AP = {ap_mean:.4f} → 기준선보다 {'높음(정보 있음)' if ap_mean > baseline else '낮음'}")

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
model.fit(X_tr, y_tr)
proba = model.predict_proba(X_te)[:, 1]
auc = roc_auc_score(y_te, proba)
print(f"\nROC-AUC (홀드아웃) = {auc:.4f}")

# 4) 표준편차 비교
print("\n=== 표준편차 큰 순서 ===")
for metric, (mean, std) in sorted(summary.items(), key=lambda x: -x[1][1]):
    print(f"  {metric:20s} std={std:.4f}")

=== 5-Fold 교차검증 결과 (평균 ± 표준편차) ===
  accuracy             = 0.8969 ± 0.0034
  precision            = 0.7232 ± 0.0117
  recall               = 0.5409 ± 0.0241
  f1                   = 0.6186 ± 0.0175
  average_precision    = 0.7248 ± 0.0191

기준선(무작위 정밀도) = 양성 비율 = 0.1547
AP = 0.7248 → 기준선보다 높음(정보 있음)

ROC-AUC (홀드아웃) = 0.9002

=== 표준편차 큰 순서 ===
  recall               std=0.0241
  average_precision    std=0.0191
  f1                   std=0.0175
  precision            std=0.0117
  accuracy             std=0.0034


**1) 지표 5종, 5-Fold 교차검증 결과**

| 지표 | 평균 ± 표준편차 | 문제1의 단일 split 값 |
|---|---|---|
| Accuracy | 0.8969 ± 0.0034 | 0.8913 |
| Precision | 0.7232 ± 0.0117 | 0.7050 |
| Recall | 0.5409 ± 0.0241 | 0.5131 |
| F1 | 0.6186 ± 0.0175 | 0.5939 |
| AP | 0.7248 ± 0.0191 | (측정 안 함) |

문제 1에서 봤던 단일 split 값들이 5-fold 평균 범위 안에 잘 들어옵니다 — 즉 그때 얻었던 결과가 "운 좋게 쉬운 fold를 뽑은 것"은 아니었다는 뜻입니다. 다만 recall이 5개 fold를 평균 내니 0.5409로, 단일 split(0.5131)보다 약간 높게 나왔네요.

**2) AP(0.7248) vs 기준선(0.1547)**

- 무작위로 찍었을 때 기대되는 정밀도가 0.1547(양성 비율)인데, 이 모델의 AP는 0.7248입니다.
- **거의 4.7배** 높습니다. 이건 모델이 "구매할 것 같은 순서로 세션을 정렬하는 능력"이 확실히 있다는 뜻입니다 — threshold 하나로 딱 자른 recall/precision보다, PR 곡선 전체를 보는 AP가 이 모델의 진짜 실력을 더 잘 보여줍니다.
- **ROC-AUC 0.9002**도 같은 이야기를 합니다. 확률 순위 자체는 꽤 잘 매기고 있다는 것이죠 (0.5=랜덤, 1.0=완벽).

**3) AP(0.7248)와 ROC-AUC(0.9002)의 차이가 왜 크게 나는가 — 이게 중요합니다**

- ROC-AUC는 "양성/음성을 섞어서 아무 짝이나 골라도 순위가 맞을 확률"이라, 다수 클래스(안 삼, 84.5%)가 압도적으로 많아도 크게 영향받지 않습니다.
- AP(=PR-AUC 유사)는 분모가 "모델이 양성이라고 예측한 것들" 위주라, **소수 클래스의 성능에 훨씬 민감**합니다.
- 그래서 불균형 데이터에서는 ROC-AUC 0.90이라는 높은 숫자만 보고 "잘한다"고 착각하기 쉬운데, AP 0.72가 좀 더 현실적인 그림입니다. (AP가 여전히 기준선보다 훨씬 높으니 이 모델이 무용지물은 아니지만, "완벽"과는 거리가 있다는 뜻이죠.)

**4) 표준편차가 가장 큰 지표: Recall (0.0241)**

이유는 지난 설명과 동일하게 **분모의 크기**입니다:

- Recall의 분모 = 실제 구매 세션 수 (전체의 15.47%뿐). Fold 하나당 양성 샘플이 약 380건 정도밖에 없습니다.
- 표본이 적으면, 그 fold에 "판별하기 쉬운 구매자"가 몰리느냐 "애매한 구매자"가 몰리느냐에 따라 recall 값이 크게 출렁입니다.
- 반면 Accuracy의 분모는 전체 샘플(다수 클래스 포함)이라 fold 간 변동이 거의 안 생깁니다 (std=0.0034로 가장 작음).

흥미로운 점은 **AP의 표준편차(0.0191)가 F1(0.0175)보다도 크게 나왔다**는 것입니다. AP도 소수 클래스 중심 지표라 recall만큼은 아니지만 fold별로 흔들리는 편입니다. 반면 precision(0.0117)은 상대적으로 안정적인데, 이건 모델이 "확신 있을 때만 구매로 찍는" 보수적인 성향(threshold=0.5에서 정밀도 우선) 때문에 예측이 fold마다 일관되게 나오는 것으로 보입니다.

---

**정리하면**: Accuracy 0.89~0.90이라는 숫자는 fold 간에 거의 안 흔들리는 안정적인 숫자지만, 그건 다수 클래스를 잘 맞추는 능력을 재는 것뿐입니다. 정작 마케팅팀이 궁금한 "구매자를 얼마나 잘 잡아내는가"(recall)는 fold마다 ±2.4%p씩 흔들리는 불안정한 숫자입니다. 이게 바로 불균형 데이터에서 accuracy 하나만 보고하면 안 되는 이유를 숫자로 보여주는 대목입니다.

In [ ]:
# [C4] 문제 3. 임계값을 훑어 운영점을 정한다
# ⌨️ 문제 3 — 용량 제약(쿠폰 500장)을 임계값으로 번역하기
import numpy as np
from sklearn.metrics import precision_score, recall_score, f1_score

# 1) 임계값 스캔 (0.10 ~ 0.70, 0.05 단위)
thresholds = np.arange(0.10, 0.71, 0.05)
rows = []
for t in thresholds:
    pred_t = (proba >= t).astype(int)
    n_pos = pred_t.sum()
    p = precision_score(y_te, pred_t, zero_division=0)
    r = recall_score(y_te, pred_t, zero_division=0)
    f = f1_score(y_te, pred_t, zero_division=0)
    rows.append({"threshold": round(t, 2), "n_positive_pred": n_pos,
                 "precision": p, "recall": r, "f1": f})

table = pd.DataFrame(rows)
print("=== 임계값 스캔 표 ===")
print(table.to_string(index=False))

# 2) F1 최적 임계값
best_row = table.loc[table["f1"].idxmax()]
print(f"\nF1 최대 임계값 = {best_row['threshold']:.2f} (F1={best_row['f1']:.4f})")

# 3) 쿠폰 500장 제약 → 임계값 역산
CAPACITY = 500
sorted_proba = np.sort(proba)[::-1]
t_capacity = sorted_proba[CAPACITY - 1]
print(f"\n쿠폰 500장 제약 임계값 = {t_capacity:.4f}")

pred_capacity = (proba >= t_capacity).astype(int)
n_selected = pred_capacity.sum()
print(f"실제 선택된 건수 = {n_selected}건")

# 4) 운영점의 실제 구매 세션 수(TP)
tp_capacity = ((pred_capacity == 1) & (y_te.values == 1)).sum()
precision_capacity = precision_score(y_te, pred_capacity, zero_division=0)
recall_capacity = recall_score(y_te, pred_capacity, zero_division=0)

print(f"\n=== 쿠폰 500장 운영점 보고 ===")
print(f"임계값 = {t_capacity:.4f}")
print(f"발송 대상 = {n_selected}건")
print(f"이 중 실제 구매 세션(TP) = {tp_capacity}건")
print(f"정밀도 = {precision_capacity:.4f}")
print(f"재현율 = {recall_capacity:.4f}")

=== 임계값 스캔 표 ===
 threshold  n_positive_pred  precision   recall       f1
      0.10              729   0.459534 0.876963 0.603060
      0.15              595   0.522689 0.814136 0.636643
      0.20              543   0.550645 0.782723 0.646486
      0.25              506   0.565217 0.748691 0.644144
      0.30              482   0.574689 0.725131 0.641204
      0.35              449   0.599109 0.704188 0.647413
      0.40              396   0.633838 0.657068 0.645244
      0.45              337   0.673591 0.594241 0.631433
      0.50              278   0.705036 0.513089 0.593939
      0.55              216   0.782407 0.442408 0.565217
      0.60              183   0.825137 0.395288 0.534513
      0.65              135   0.874074 0.308901 0.456480
      0.70              112   0.875000 0.256545 0.396761

F1 최대 임계값 = 0.35 (F1=0.6474)

쿠폰 500장 제약 임계값 = 0.2683
실제 선택된 건수 = 500건

=== 쿠폰 500장 운영점 보고 ===
임계값 = 0.2683
발송 대상 = 500건
이 중 실제 구매 세션(TP) = 284건
정밀도 = 0.5680
재현율 = 0.7435


F1 최적점과 쿠폰 500장 제약점이 이번에는 **꽤 다르게** 나왔습니다. 이게 실무에서 흔히 벌어지는 일이고, 오히려 더 좋은 학습 포인트입니다.

**1) F1 최적 임계값(0.35) vs 쿠폰 제약 임계값(0.2683) — 왜 다른가**

| | 임계값 | 양성 예측 | 정밀도 | 재현율 | F1 |
|---|---|---|---|---|---|
| F1 최적점 | 0.35 | 449건 | 0.5991 | 0.7042 | **0.6474** |
| 쿠폰 500장 | 0.2683 | 500건 | 0.5680 | 0.7435 | 0.6472 |

- 흥미롭게도 F1 값 자체는 거의 같습니다(0.6474 vs 0.6472). F1 곡선이 임계값 0.20~0.40 구간에서 상당히 평평(0.641~0.647)하기 때문입니다.
- 즉, **"F1 관점에서는 449건을 뽑든 500건을 뽑든 거의 차이가 없다"**는 뜻입니다. 그래서 쿠폰 500장이라는 운영 제약을 그냥 따라도 성능 손실이 거의 없습니다 — 이건 좋은 소식입니다. 굳이 F1 최적점(449건)에 맞춰 쿠폰을 51장 아낄 필요가 없다는 뜻이죠.

**2) 쿠폰 500장 운영점 — 마케팅팀 보고**

> "쿠폰 500장을 확률 상위 순으로 배포하면, 예측 확률 약 26.8% 이상인 세션이 선택됩니다(임계값 0.2683). 이 500건 중 실제로 구매할 세션은 **284건(정밀도 56.8%)**입니다. 이는 테스트셋 전체 구매 세션 382건 중 **74.4%(재현율)**를 이 쿠폰 캠페인으로 포착한다는 의미입니다. 나머지 216장(500-284)은 실제로는 구매하지 않을 세션에 쓰이지만, 216명이 아닌 구매자 46%(382×0.2565)를 쿠폰 없이 놔두는 대신 잡아내는 트레이드오프입니다."

**3) 임계값 0.5(기본값) 대비 임계값 0.2683으로 낮춘 효과가 극명합니다**

- 문제 1~2에서 봤던 기본 임계값 0.5의 재현율은 0.51~0.54였습니다.
- 쿠폰 500장 제약을 따라 임계값을 0.2683으로 낮추자, 재현율이 **0.7435**로 뛰었습니다. 정밀도는 0.705 → 0.568로 떨어졌지만, "구매 세션을 최대한 많이 포착"이라는 마케팅팀의 원래 목적에는 이 운영점이 훨씬 부합합니다.
- 이게 바로 문제 1에서 지적했던 "정확도 89%가 착시"였던 문제의 실질적 해법입니다 — 기본 임계값 0.5를 그대로 쓸 이유가 없었던 것이고, 용량 제약(쿠폰 500장)이 오히려 더 합리적인 임계값을 알려준 셈입니다.

**4) 표에서 드러나는 또 하나의 패턴**

- 정밀도는 임계값이 오를수록 단조증가(0.46→0.88)하지만, 재현율은 단조감소(0.88→0.26)합니다 — 이건 항상 성립하는 이론적 관계입니다.
- F1은 그 사이 어딘가에서 봉우리를 만드는데, 이번 데이터에서는 그 봉우리가 넓고 평평해서(0.30~0.40 구간 F1이 0.641~0.647로 거의 붙어 있음) 정확한 최적점 하나를 고집할 필요가 없다는 걸 보여줍니다. 이런 경우 실무에서는 "F1이 아니라 운영 제약(쿠폰 수량)"이 최종 결정을 좌우하는 게 더 합리적입니다.

In [ ]:
# [C5] 문제 4. class_weight 보정, 채택할 것인가
# ⌨️ 문제 4 — 보정 전후를 같은 방식으로 비교
from sklearn.metrics import confusion_matrix

def cv_report(model, label):
    results = cross_validate(model, X, y, cv=cv, scoring=scoring)
    row = {"model": label}
    for metric in scoring:
        scores = results[f"test_{metric}"]
        row[metric] = f"{scores.mean():.4f} ± {scores.std():.4f}"
    return row

# 1) 기본 모델 vs balanced 모델, 같은 방식으로 CV
model_base = RandomForestClassifier(n_estimators=300, min_samples_leaf=20, random_state=42)
row_base = cv_report(model_base, "기본(base)")

model_bal = RandomForestClassifier(n_estimators=300, min_samples_leaf=20, random_state=42,
                                     class_weight="balanced")
row_bal = cv_report(model_bal, "balanced")

# 2) 비교표
comp_table = pd.DataFrame([row_base, row_bal])
print("=== 기본 vs class_weight=balanced 비교 ===")
print(comp_table.to_string(index=False))

# 3) 홀드아웃 혼동행렬 대조 (threshold=0.5 그대로)
model_base.fit(X_tr, y_tr)
pred_base = model_base.predict(X_te)
tn_b, fp_b, fn_b, tp_b = confusion_matrix(y_te, pred_base).ravel()

model_bal.fit(X_tr, y_tr)
pred_bal = model_bal.predict(X_te)
tn_a, fp_a, fn_a, tp_a = confusion_matrix(y_te, pred_bal).ravel()

print("\n=== 홀드아웃 혼동행렬 대조 ===")
print(f"{'':12s} {'TN':>6s} {'FP':>6s} {'FN':>6s} {'TP':>6s}")
print(f"{'기본':12s} {tn_b:6d} {fp_b:6d} {fn_b:6d} {tp_b:6d}")
print(f"{'balanced':12s} {tn_a:6d} {fp_a:6d} {fn_a:6d} {tp_a:6d}")
print(f"\nFN 변화: {fn_b} → {fn_a} ({fn_a - fn_b:+d})")
print(f"TP 변화: {tp_b} → {tp_a} ({tp_a - tp_b:+d})")
print(f"FP 변화: {fp_b} → {fp_a} ({fp_a - fp_b:+d})")

=== 기본 vs class_weight=balanced 비교 ===
   model        accuracy       precision          recall              f1 average_precision
기본(base) 0.8969 ± 0.0034 0.7232 ± 0.0117 0.5409 ± 0.0241 0.6186 ± 0.0175   0.7248 ± 0.0191
balanced 0.8691 ± 0.0065 0.5529 ± 0.0148 0.8092 ± 0.0234 0.6568 ± 0.0145   0.7175 ± 0.0150

=== 홀드아웃 혼동행렬 대조 ===
                 TN     FP     FN     TP
기본             2002     82    186    196
balanced       1847    237     83    299

FN 변화: 186 → 83 (-103)
TP 변화: 196 → 299 (+103)
FP 변화: 82 → 237 (+155)


실제 데이터에서 예상했던 패턴이 그대로 나왔습니다. 판단해 보겠습니다.

**1) 지표 변화 요약**

| 지표 | 기본 | balanced | 변화 |
|---|---|---|---|
| Accuracy | 0.8969 | 0.8691 | -0.0278 |
| Precision | 0.7232 | 0.5529 | -0.1703 |
| Recall | 0.5409 | 0.8092 | **+0.2683** |
| F1 | 0.6186 | 0.6568 | +0.0382 |
| **AP** | **0.7248** | **0.7175** | **-0.0073** |

**2) 혼동행렬 변화**

- FN 186 → 83 (**103건 덜 놓침**)
- TP 196 → 299 (**103건 더 잡음**)
- FP 82 → 237 (**155건 더 헛발질**)

recall이 0.54 → 0.81로 크게 뛰어서 F1도 개선됐지만, 그 대가로 FP가 3배 가까이 늘었습니다.

**3) 핵심 판단 근거: AP가 소폭 하락했습니다 (0.7248 → 0.7175)**

이게 결정을 가르는 지점입니다. AP는 임계값과 무관하게 "모델이 확률을 얼마나 잘 매기는가"를 재는 지표인데, class_weight="balanced"를 적용해도 이 순위 품질은 개선되지 않았고 오히려 살짝 나빠졌습니다. 즉:

- balanced 모델이 보여준 recall 상승(0.54→0.81)은 **모델이 더 똑똑해진 게 아니라, 트리 분할 기준이 소수 클래스 쪽으로 밀리면서 "구매"라고 찍는 문턱이 낮아진 효과**에 가깝습니다.
- 이건 문제 3에서 임계값을 0.5 → 0.27로 낮췄을 때 벌어진 일과 사실상 같은 방향의 효과입니다. 문제 3에서 임계값 0.2683으로 낮췄을 때: 재현율 0.7435, 정밀도 0.5680, 발송 500건 — 지금 balanced 모델의 재현율 0.8092, 정밀도 0.5529와 상당히 비슷한 영역입니다.

**4) 결론: class_weight="balanced"는 채택하지 않습니다**

이유는 세 가지입니다.

- **같은 효과를 임계값 조정으로 이미 얻을 수 있습니다.** 문제 3에서 기본 모델의 확률에 임계값만 낮춰서 recall 0.74, 쿠폰 500장이라는 운영 제약에 정확히 맞춘 지점을 찾았습니다. balanced 모델을 새로 학습시킬 필요 없이, 기본 모델 하나로 이 스펙트럼 전체(임계값 0.10~0.70)를 자유롭게 오갈 수 있습니다.
- **AP가 하락했다는 건 모델의 근본적인 순위 품질이 나빠졌다는 신호입니다.** balanced 모델을 채택하면, 그 모델이 내놓는 확률값 자체가 왜곡되어(원래 클래스 비율을 반영하지 않음) 이후 캘리브레이션이나 다른 임계값 실험을 할 때 해석이 더 복잡해집니다.
- **운영 유연성이 떨어집니다.** 쿠폰 수량이 500장에서 300장, 700장으로 바뀔 때마다 balanced 모델은 재학습하거나 임계값을 다시 스캔해야 하는데, 기본 모델은 이미 가진 확률에서 임계값만 옮기면 됩니다. 즉 "기본 모델 + 임계값 튜닝"이 더 단순하고 통제하기 쉬운 해법입니다.

**언제 balanced를 쓸 만한가** — 참고로, 이번처럼 양성 비율이 15% 수준일 때는 트리가 소수 클래스를 학습에서 완전히 무시하지 않기 때문에 임계값 조정만으로 충분합니다. balanced가 진짜 필요한 경우는 양성 비율이 1% 미만처럼 극단적으로 낮아서, 기본 임계값 근처에서 아무리 낮춰도 트리 분할 자체가 소수 클래스를 거의 못 보는 상황입니다.

---

**최종 정리**: 이번 실험에서 class_weight="balanced"는 recall과 F1을 개선했지만, 그 효과가 "학습된 모델의 순위 능력 향상"이 아니라 "결정 경계 이동"이었고, 이건 문제 3의 임계값 조정과 동일한 효과를 다른 방식으로 낸 것뿐입니다. AP가 오히려 하락했다는 점을 근거로, 이번 프로젝트에서는 **class_weight를 채택하지 않고, 기본 모델 + 임계값 튜닝(문제 3의 방식)을 유지**하는 것이 더 합리적인 선택입니다.

In [ ]:
# [C6] 문제 5. 모델 카드 v4 완성 — 오늘의 제출물
# ⌨️ 문제 5 — 임계값 표를 파일로 남기기

# 문제 3에서 만든 변수명은 'table' 입니다
thr_tbl = table.copy()

thr_tbl.to_csv("threshold_table_v4.csv", index=False)

# 다시 읽어 확인
check = pd.read_csv("threshold_table_v4.csv")
print(check.to_string(index=False))

 threshold  n_positive_pred  precision   recall       f1
      0.10              729   0.459534 0.876963 0.603060
      0.15              595   0.522689 0.814136 0.636643
      0.20              543   0.550645 0.782723 0.646486
      0.25              506   0.565217 0.748691 0.644144
      0.30              482   0.574689 0.725131 0.641204
      0.35              449   0.599109 0.704188 0.647413
      0.40              396   0.633838 0.657068 0.645244
      0.45              337   0.673591 0.594241 0.631433
      0.50              278   0.705036 0.513089 0.593939
      0.55              216   0.782407 0.442408 0.565217
      0.60              183   0.825137 0.395288 0.534513
      0.65              135   0.874074 0.308901 0.456480
      0.70              112   0.875000 0.256545 0.396761


## 모델 카드 v4 — 구매 전환 예측

- 데이터: UCI Online Shoppers (12,330 세션, 수치형 10개 열, 양성 15.5%)
- 문제 유형: 분류 (불균형)
- 검증 방식: StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
- 모델: RandomForest(n_estimators=300, min_samples_leaf=20)
- **주 지표와 선택 근거**: F1과 AP를 정확도와 함께 본다. 정확도(0.8969)는 다수 클래스(안 삼, 84.5%)만 잘 맞춰도 쉽게 높게 나오는 지표라, 소수 클래스(구매)를 얼마나 잡아내는지는 전혀 알려주지 않는다. 실제로 정확도 0.89 상태에서도 재현율은 0.54에 그쳐 구매자의 절반 가까이를 놓쳤다. F1은 정밀도·재현율을 함께 반영하고, AP는 임계값에 무관하게 확률 순위 품질을 재기 때문에 운영점을 정하기 전 모델의 근본 실력을 보는 데 쓴다.
- **성능(CV, 평균 ± 표준편차)**
  - 정확도 0.8969 ± 0.0034 / 정밀도 0.7232 ± 0.0117 / 재현율 0.5409 ± 0.0241 / F1 0.6186 ± 0.0175 / AP 0.7248 ± 0.0191
  - **양성 비율 0.1547와 AP 0.7248 비교** → AP가 기준선(무작위 정밀도)보다 약 4.7배 높아, 모델이 구매 가능성 순으로 세션을 유의미하게 정렬하고 있음을 확인.
- **혼동행렬(임계값 0.5): TN 2002 / FP 82 / FN 186 / TP 196**
  - **놓친 구매 세션 186건 (48.7%)** ← 정확도만 봤을 때 보이지 않던 숫자
- **운영점: 임계값 0.2683 (용량 제약 500건)**
  - **그 지점의 정밀도 0.5680 / 재현율 0.7435 → 실제 구매 세션 284건 포함**
  - **선택 근거**: F1 최적 임계값(0.35, F1=0.6474)과 용량 제약 임계값(0.2683, F1=0.6472)의 F1 값은 거의 동일하다 — 임계값 0.20~0.40 구간에서 F1 곡선이 평평하기 때문이다. 즉 F1 관점의 손실이 미미한 반면, 쿠폰 500장이라는 운영 제약(마케팅팀의 실제 가용 자원)은 고정되어 있으므로 F1 최적점을 고집하지 않고 용량 제약을 최종 기준으로 삼았다. 예산이 정해진 캠페인에서는 지표 최적화보다 제약 충족이 실행 가능한 결정이다.
- **class_weight 보정: 채택 안 함 — 근거**: balanced 모델은 재현율을 0.5409→0.8092로 크게 올렸지만 AP는 0.7248→0.7175로 오히려 하락했다. 이는 모델의 확률 순위 품질 자체가 개선된 게 아니라 결정 경계가 소수 클래스 쪽으로 이동한 효과임을 시사한다. 동일한 재현율 상승은 기본 모델의 임계값을 0.5→0.27 수준으로 낮추는 것만으로도 재현율 0.7435까지 달성 가능했다(운영점 참조). 재학습 없이 임계값만 조정하면 되므로, 운영 유연성과 확률 해석의 단순성 면에서 기본 모델 유지가 더 합리적이다.
- **한계 & 다음 단계**: (1) 이 모델은 "구매 확률"을 예측할 뿐, 쿠폰을 줬을 때 실제로 구매 행동이 바뀌는 "개입 효과(uplift)"는 측정하지 않았다 — 상위 500명 중 상당수는 쿠폰 없이도 구매했을 수 있다. (2) 범주형 8개 열(방문자 유형, 요일, 브라우저 등)을 사용하지 않아 정보 손실이 있을 수 있다. (3) 운영점(임계값 0.2683)은 이번 홀드아웃 하나에서 정한 값이라, 실제 배포 전에는 교차검증의 OOF(out-of-fold) 예측이나 별도 검증셋에서 재확인이 필요하다.
- **AI 사용 내역**: Claude에게 각 문제의 코드 완성(혼동행렬, cross_validate 다중 지표, 임계값 스캔, class_weight 비교)을 요청했고, 로직은 합성 데이터로 먼저 검증받았다(환경 네트워크 제약으로 UCI 실데이터 직접 실행은 불가). 실제 UCI 데이터 실행 결과는 직접 실행 후 붙여넣어 Claude에게 해석과 판단 근거 검토를 요청했다.